In [0]:
%run "../00_config"

In [0]:
import requests
import json
import os
from datetime import datetime, timezone
import time
from pyspark.sql import functions as F

##Fetch Saudi holidays

In [0]:
#Datbricks Community Edition wont allo direct api request
#so I had to upload the data manually

# def fetch_holidays(api_key: str, year: int) -> list:
#     url = "https://calendarific.com/api/v2/holidays"
#     params = {
#         "api_key": api_key,
#         "country": "SA",
#         "year":    year
#     }
#     try:
#         r = requests.get(url, params=params, timeout=15)
#         r.raise_for_status()
#         data = r.json()
#         holidays = data.get("response", {}).get("holidays", [])
#         print(f"  Fetched {len(holidays)} holidays for {year}")
#         return holidays
#     except Exception as e:
#         print(f"  Error: {e}")
#         return []

files = dbutils.fs.ls(HOLIDAYS_VOLUME)
years = []
for f in files:
  year = f.name.split("_")[1].split(".")[0]
  years.append(int(year))

all_holidays = [] 

for year in years:
    with open(f"{HOLIDAYS_VOLUME}/holidays_{year}.json", "r") as f:
        data = json.load(f)
  
    holidays = data.get("response", {}).get("holidays", [])
    print(f"  Fetched {len(holidays)} holidays for {year}")
    
    for h in holidays:
        all_holidays.append({
            "holiday_name":        h.get("name", ""),
            "holiday_description": h.get("description", ""),
            "holiday_date":        h.get("date", {}).get("iso", ""),
            "holiday_type":        ", ".join(h.get("type", [])),
            "country":             "SA",
            "year":                str(year),
            "_snapshot_date":      SNAPSHOT_DATE,
            "_ingested_at":        datetime.now(timezone.utc).strftime("%d-%m-%Y %H:%M:%S")
        })
    

print(f"\nTotal holiday records: {len(all_holidays)}")

##Write to Bronze

In [0]:
df_bronze_holidays = spark.createDataFrame(all_holidays)

(df_bronze_holidays
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_HOLIDAYS)
)

print(f"✅ {df_bronze_holidays.count()} holidays written to {BRONZE_HOLIDAYS}")

##Validate

In [0]:
spark.table(BRONZE_HOLIDAYS) \
    .select("holiday_name", "holiday_date", "holiday_type", "year") \
    .orderBy("holiday_date") \
    .show(20, truncate=50)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.bronze.bronze_holidays